# 南水北调中线工程参数提取结果可视化

本 Notebook 对分割与检测模型提取的工程参数进行可视化分析，包括：
- 渠道宽度沿程变化
- 建筑物空间分布
- 交互式地图
- 统计汇总报表

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
import folium
from IPython.display import display

from seehydro.export.visualization import create_folium_map, plot_width_profile, mask_to_rgb
from seehydro.export.report import generate_summary_report, save_report

# 设置中文字体（优先 SimHei，回退 DejaVu Sans）
matplotlib.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["figure.dpi"] = 120

# 输出目录
OUTPUT_DIR = Path("outputs")
VECTOR_DIR = OUTPUT_DIR / "vectors"
FIGURE_DIR = OUTPUT_DIR / "figures"
REPORT_DIR = OUTPUT_DIR / "reports"

print("环境初始化完成")


## 1. 加载提取结果

从 `outputs/vectors/` 读取各类 GeoJSON 文件。

In [ ]:
# 渠道宽度剖面
canal_gdf = None
try:
    canal_gdf = gpd.read_file(VECTOR_DIR / "canal_width.geojson")
    print(f"渠道宽度点数: {len(canal_gdf)}")
    display(canal_gdf.head())
except FileNotFoundError:
    print("[警告] canal_width.geojson 不存在，跳过渠道宽度分析")

# 桥梁
bridges_gdf = None
try:
    bridges_gdf = gpd.read_file(VECTOR_DIR / "bridges.geojson")
    print(f"桥梁数量: {len(bridges_gdf)}")
except FileNotFoundError:
    print("[警告] bridges.geojson 不存在")

# 倒虹吸
siphons_gdf = None
try:
    siphons_gdf = gpd.read_file(VECTOR_DIR / "siphons.geojson")
    print(f"倒虹吸数量: {len(siphons_gdf)}")
except FileNotFoundError:
    print("[警告] siphons.geojson 不存在")

# 渡槽
aqueducts_gdf = None
try:
    aqueducts_gdf = gpd.read_file(VECTOR_DIR / "aqueducts.geojson")
    print(f"渡槽数量: {len(aqueducts_gdf)}")
except FileNotFoundError:
    print("[警告] aqueducts.geojson 不存在")

# 闸门
gates_gdf = None
try:
    gates_gdf = gpd.read_file(VECTOR_DIR / "gates.geojson")
    print(f"闸门数量: {len(gates_gdf)}")
except FileNotFoundError:
    print("[警告] gates.geojson 不存在")

# 合并所有建筑物（用于地图显示）
structure_gdfs = [g for g in [siphons_gdf, aqueducts_gdf, gates_gdf] if g is not None]
structures_gdf = (
    gpd.GeoDataFrame(pd.concat(structure_gdfs, ignore_index=True))
    if structure_gdfs else None
)
print(f"
合并建筑物总数: {len(structures_gdf) if structures_gdf is not None else 0}")


## 2. 渠道宽度沿程分析

沿渠道中心线每 50m 采样一次水面宽度，展示宽度沿程变化趋势。

In [ ]:
if canal_gdf is not None and "width_m" in canal_gdf.columns:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    fig = plot_width_profile(
        canal_gdf,
        output_path=FIGURE_DIR / "canal_width_profile.png",
        title="南水北调中线渠道水面宽度沿程变化",
    )
    plt.show()

    # 统计信息
    width_stats = canal_gdf["width_m"].describe()
    print("
=== 渠道宽度统计 ===")
    print(f"采样点数:  {int(width_stats['count'])}")
    print(f"最小宽度:  {width_stats['min']:.1f} m")
    print(f"最大宽度:  {width_stats['max']:.1f} m")
    print(f"平均宽度:  {width_stats['mean']:.1f} m")
    print(f"中位宽度:  {width_stats['50%']:.1f} m")
    print(f"标准差:    {width_stats['std']:.1f} m")
else:
    print("[跳过] 无渠道宽度数据")


## 3. 建筑物空间分布

以经纬度坐标绘制各类交叉建筑物的平面分布图。

In [ ]:
all_gdfs = {
    "桥梁": bridges_gdf,
    "倒虹吸": siphons_gdf,
    "渡槽": aqueducts_gdf,
    "闸门": gates_gdf,
}
valid_gdfs = {k: v for k, v in all_gdfs.items() if v is not None and len(v) > 0}

if valid_gdfs:
    fig, ax = plt.subplots(figsize=(16, 6))

    # 渠道中心线（若有）
    if canal_gdf is not None:
        canal_gdf.plot(
            ax=ax, color="#3498db", linewidth=1.5,
            label="渠道", alpha=0.6, markersize=2,
        )

    # 各类建筑物
    markers = ["o", "s", "^", "D", "P", "*", "X"]
    for idx, (label, gdf) in enumerate(valid_gdfs.items()):
        geom_type = gdf.geometry.geom_type.iloc[0]
        if geom_type in ("Point", "MultiPoint"):
            ax.scatter(
                gdf.geometry.x, gdf.geometry.y,
                label=f"{label} (n={len(gdf)})",
                s=60, zorder=5,
                marker=markers[idx % len(markers)],
            )
        else:
            gdf.plot(ax=ax, label=f"{label} (n={len(gdf)})")

    ax.set_xlabel("经度", fontsize=12)
    ax.set_ylabel("纬度", fontsize=12)
    ax.set_title("南水北调中线交叉建筑物空间分布", fontsize=14)
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / "structures_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图表已保存至 {FIGURE_DIR / 'structures_distribution.png'}")
else:
    print("[跳过] 无建筑物数据可用于绘图")


## 4. 交互地图

基于 Folium 生成可缩放的交互地图，各图层可独立开关。

In [ ]:
folium_map = create_folium_map(
    canal_gdf=canal_gdf,
    bridges_gdf=bridges_gdf,
    structures_gdf=structures_gdf,
    zoom_start=9,
)

# 在 Notebook 中内嵌显示
display(folium_map)


## 5. 统计报表

汇总渠道、桥梁、倒虹吸、渡槽、闸门的关键参数统计。

In [ ]:
# 构造渠道参数字典
canal_params = None
if canal_gdf is not None and "width_m" in canal_gdf.columns:
    canal_params = {
        "mean_width_m": round(canal_gdf["width_m"].mean(), 1),
        "width_profile": canal_gdf["width_m"].tolist(),
    }
    if "berm_width_m" in canal_gdf.columns:
        canal_params["mean_berm_width_m"] = round(canal_gdf["berm_width_m"].mean(), 1)

summary_df = generate_summary_report(
    canal_params=canal_params,
    bridges=bridges_gdf,
    siphons=siphons_gdf,
    aqueducts=aqueducts_gdf,
    gates=gates_gdf,
)

print("=== 南水北调中线工程参数汇总 ===")
display(summary_df)


## 6. 导出结果

将统计报表保存为 CSV / Excel，交互地图保存为 HTML。

In [ ]:
# 保存统计报表（CSV + Excel）
if not summary_df.empty:
    saved_paths = save_report(summary_df, output_dir=REPORT_DIR, name="工程参数汇总")
    for p in saved_paths:
        print(f"报表已保存: {p}")
else:
    print("[跳过] 汇总表为空，未保存报表")

# 保存交互地图 HTML
map_path = REPORT_DIR / "interactive_map.html"
map_path.parent.mkdir(parents=True, exist_ok=True)
folium_map.save(str(map_path))
print(f"交互地图已保存: {map_path}")

print("
=== 所有结果导出完成 ===")
print(f"向量数据: {VECTOR_DIR}")
print(f"图表:     {FIGURE_DIR}")
print(f"报表:     {REPORT_DIR}")
